# 24-modulation training on Colab's GPU

Trains VGG and ResNet on all 24 modulations, on a free Colab GPU. The data was
already prepared on the laptop (24 classes x 20,000 examples) and bundled into
`colab_payload.tar`, which should already be sitting in your Google Drive.

**Before you run anything:**
1. Confirm `colab_payload.tar` (~3.9 GB) is in the top level of your Google Drive
   ("My Drive").
2. Turn the GPU on: menu **Runtime -> Change runtime type -> Hardware accelerator = GPU**.

Then run the cells top to bottom. If your connection drops mid-run, just reopen this
tab — Colab reconnects the same kernel automatically (that's the whole reason we
switched from the CLI to this notebook).

## 1. Check we actually got a GPU
If this errors or shows no GPU, redo Runtime -> Change runtime type.

In [ ]:
!nvidia-smi

## 2. Connect Google Drive
A pop-up will ask permission — that lets Colab read `colab_payload.tar` from your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PAYLOAD = '/content/drive/MyDrive/colab_payload.tar'
import os
assert os.path.isfile(PAYLOAD), f'Did not find {PAYLOAD} — check it is uploaded to the top level of My Drive.'
print(f'Found: {PAYLOAD}  ({os.path.getsize(PAYLOAD)/1e9:.2f} GB)')

## 3. Extract the bundle onto Colab's local disk
`colab_payload.tar` already contains `scripts/` and the prepared 24-class arrays
(`prepared/X_train.npy` etc. — 480,000 examples, already sampled and normalized).
No need to touch the 19.5 GB raw dataset at all.

In [ ]:
import tarfile, os, time

ROOT = '/content/proj'
os.makedirs(ROOT, exist_ok=True)

t = time.time()
with tarfile.open(PAYLOAD) as tf:
    tf.extractall(ROOT)
print(f'Extracted in {time.time()-t:.0f}s. Contents: {sorted(os.listdir(ROOT))}')
print('prepared/:', sorted(os.listdir(f'{ROOT}/prepared')))

## 4. Train both models (VGG and ResNet)
A few minutes each on the GPU. Both train on the same prepared data from the bundle.

In [ ]:
!cd {ROOT} && python scripts/train.py --model vgg
!cd {ROOT} && python scripts/train.py --model resnet

## 5. Evaluate both + high-SNR score
Confusion matrix and per-class precision/recall for each model, then the paper-style
clean-signal number for both side by side (`high_snr.py` scores both at once).

In [ ]:
!cd {ROOT} && python scripts/evaluate.py --model vgg
!cd {ROOT} && python scripts/evaluate.py --model resnet
!cd {ROOT} && python scripts/high_snr.py

## 6. Save the results back to Drive
Colab wipes local disk when the session ends, so we copy the trained models and
figures into `outputs/` at the top level of your Drive to keep them.

In [ ]:
import shutil, os
OUT = '/content/drive/MyDrive/outputs'
for sub in ['results', 'models']:
    src = f'{ROOT}/{sub}'
    if os.path.isdir(src):
        dst = f'{OUT}/{sub}'
        if os.path.isdir(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print('saved', dst)
print('All results are in your Drive under outputs/.')